# A.5 答錯是沒找到還是沒用好？


三家書店只答對一家，可能是沒有找到公告，也可能找到後讀錯。先為每題列出所需來源，再保存實際取回的編號與答案，就能把兩站責任分開。

下面是三份人工診斷紀錄。hit表示所需來源出現在取回清單，correct表示答案符合固定真值。甲兩項都對，乙找到但答錯，丙既沒找到也沒答對。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
examples = [
    {"question": "甲店地址", "hit": True, "correct": True},
    {"question": "乙店地址", "hit": True, "correct": False},
    {"question": "丙店地址", "hit": False, "correct": False},
]
count = len(examples)
recall = sum(e["hit"] for e in examples) / count
accuracy = sum(e["correct"] for e in examples) / count
print("有正確來源", recall)
print("答案正確", accuracy)
for e in examples:
    if not e["correct"]:
        print(e["question"], "已找到但答錯" if e["hit"] else "尚未找到所需來源")

有正確來源 0.6666666666666666
答案正確 0.3333333333333333
乙店地址 已找到但答錯
丙店地址 尚未找到所需來源


有正確來源2/3，答案正確1/3。這裡每題只有一份指定來源，來源命中分母也是題數；需要多份來源的題目要另說「至少找到一份」還是「全部找到」。答案判準也預先固定，不能只看回答流暢。

先處理乙：核對最終提示是否完整保留公告，再查地址與引用。丙先查文件、查詢和排序。可以另把正確公告直接提供給失敗題，固定模型重新回答；若這時答對，檢索很可能是瓶頸，但提示長度或其他資料也變時要記下，不能當絕對因果證明。

只把乙的correct改True，來源比例仍2/3，答案變2/3。這個紙上變化發生在使用資料，不是多找了一份文件。後面的長上下文與對話題也沿這個區分保存中途資料。

<details>
<summary>補充：找到來源，仍可能用錯的實例</summary>

舊小世界實驗固定同一12個新店名的問題，每題只有一份所需公告，兩種條件都生成全部12題。這不是上方三筆人工紀錄。回答協議分兩支：有所問公告時，須答地址與指定來源，例如[A.2](https://birdhackor.github.io/tiny-perceptron-vlm/A.2.html)的 `K017` 應答 `C2[D1]`；沒有所問公告時，才答 `UNKNOWN`。兩支都算依回答協議答對，但 `UNKNOWN` 不是給出真實地址。

| 當次上下文 | 所問公告在上下文 | 地址與引用符合真值 | 依回答協議答對 |
| --- | --- | --- | --- |
| 直接給正確公告 | 12／12 | 5／12 | 5／12 |
| 實際檢索取回 | 9／12 | 3／12 | 6／12 |

三欄分母都是完整12道問題。檢索找到九份所問公告，卻只有三題答出正確地址與來源；另外三題沒有找到公告而答 `UNKNOWN`，使協議成功數成為六。因此「6／12依協議答對」不能當成六題地址正確。

直接給正確公告，仍有七題未答對，表示這次不只有檢索瓶頸。實際取回哪些公告、模型如何回答，仍要逐題並排查看；不能只按文字流暢程度判定使用成功。完整配方及原始生成見[正式報告](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/rag.json)，這些有限公告題不替一般長文件問答填能力分數。

</details>
